## ▶ Colab setup — GitHub repo + dataset from Google Drive

Run this cell **first** on Google Colab. It clones the GitHub repo, installs the
`phytolabs` package, mounts your Drive, and unzips + reshapes the Leaf-rust
dataset into `data/{train,val}/{healthy,rust}` so the rest of the notebook runs
on **real data**. Edit `ZIP_PATH` if your zip lives elsewhere in Drive.

On a non-Colab machine this cell is a harmless no-op (the notebook then falls
back to the synthetic smoke-test dataset).

In [ ]:
# === Colab setup: GitHub repo + dataset from Google Drive ===================
import os, sys
from pathlib import Path

REPO_URL = "https://github.com/Adian17/PhytoLabs.git"
ZIP_PATH = "/content/drive/MyDrive/ece186/WheatLeafRust.zip"  # <-- adjust if needed

if "google.colab" in sys.modules:
    # 1) Clone the repo + install the package.
    if not os.path.isdir("/content/PhytoLabs"):
        !git clone -q $REPO_URL /content/PhytoLabs
    %cd /content/PhytoLabs
    !pip -q install -e .
    # Editable install / _setup aren't importable mid-kernel; add paths explicitly.
    for _p in ("/content/PhytoLabs/src", "/content/PhytoLabs/notebooks"):
        if _p not in sys.path:
            sys.path.insert(0, _p)

    # 2) Mount Drive + unzip the dataset (only the first time).
    if not (Path("data/raw").exists() and any(Path("data/raw").iterdir())):
        from google.colab import drive
        drive.mount("/content/drive")
        !rm -rf data/raw && mkdir -p data/raw
        !unzip -q "$ZIP_PATH" -d data/raw

    # 3) Reshape control/diseased -> data/{train,val}/{healthy,rust} (only once).
    if not (Path("data/train/rust").exists() and any(Path("data/train/rust").glob("*"))):
        raw = Path("data/raw")
        def _find(name):
            cands = [d for d in raw.rglob("*") if d.is_dir() and d.name.lower() == name]
            if not cands:
                raise FileNotFoundError(f"No '{name}' folder under data/raw — check the zip layout.")
            train_cands = [d for d in cands if "train" in str(d).lower()]
            return str((train_cands or cands)[0])
        H, R = _find("control"), _find("diseased")
        print("healthy <-", H, "\nrust    <-", R)
        !python -m scripts.reshape_data --healthy-src "$H" --rust-src "$R" --out data --val-fraction 0.2
    print("dataset:", {c: len(list(Path("data/train", c).glob("*"))) for c in ("healthy", "rust")})

# Stage 1 -> 2 — Per-image features

Turn the segmentation masks into quantitative, image-level features:
- `lesion_area_fraction` — rust pixels / leaf pixels
- `blob_count` — number of distinct lesions
- `blob_size_mean/std/max` — lesion size distribution
- `blob_density` — lesions per 10k leaf pixels

In [ ]:
from _setup import DATA_DIR, ARTIFACTS_DIR, ensure_dataset
import numpy as np
import matplotlib.pyplot as plt
from phytolabs import segmentation, pipeline, viz
from phytolabs.features import FEATURE_NAMES

data_dir = ensure_dataset()
gmm_path = ARTIFACTS_DIR / 'gmm.joblib'
if gmm_path.exists():
    leaf_gmm = segmentation.LeafGMM.load(gmm_path)
else:
    leaf_gmm = pipeline.fit_gmm_from_dir(data_dir / 'train')
    leaf_gmm.save(gmm_path)

## Build a feature table for the training split

In [ ]:
X, y, paths = pipeline.build_feature_table(data_dir / 'train', leaf_gmm)
print('X shape:', X.shape, '| positives (rust):', int(y.sum()))
import numpy as np
for name, col in zip(FEATURE_NAMES, X.T):
    print(f'{name:22s} healthy_mean={col[y==0].mean():.3f}  rust_mean={col[y==1].mean():.3f}')

## Feature distributions by class

Good features separate healthy (blue) from rust (orange).

In [ ]:
viz.plot_feature_histograms(X, y, FEATURE_NAMES)
plt.show()

## Save feature tables (train + val) for Stage 2

In [ ]:
for split in ('train', 'val'):
    Xs, ys, ps = pipeline.build_feature_table(data_dir / split, leaf_gmm)
    out = ARTIFACTS_DIR / f'features_{split}.npz'
    np.savez(out, X=Xs, y=ys, paths=np.array(ps, dtype=object))
    print(f'{split}: {Xs.shape} -> {out}')